In [ ]:
# ---------------------------------------------------------------------------
# Histogram visualisation with material-class overlays
# ---------------------------------------------------------------------------
# This script loads intensity histogram data exported from a micro-CT image
# analysis tool and produces a publication-ready plot of the gray-value distribution
# for a single scan session. 
# Three CSV files are expected: one containing the full histogram of the NLM-filtered
# volume, and two containing the per-class histograms for root and sand
# voxels respectively. The script smooths the full histogram using cubic
# spline interpolation and overlays shaded bands that mark the gray-value
# ranges corresponding to each material class, allowing visual validation
# of the segmentation thresholds chosen during the pipeline.
# ---------------------------------------------------------------------------

import pandas as pd
import matplotlib.pyplot as plt
from scipy.interpolate import interp1d
import numpy as np

# ---------------------------------------------------------------------------
# Configuration — replace these values before running
# ---------------------------------------------------------------------------
# Absolute paths to the three input CSV files for the session being plotted.
# Each CSV must contain two columns: gray-value intensity and pixel count.
histogram_csv = "/path/to/histogram_filtered.csv"   # Full NLM-filtered volume histogram
root_csv      = "/path/to/root.csv"                 # Histogram of root-class voxels
sand_csv      = "/path/to/sand.csv"                 # Histogram of sand-class voxels

# Absolute path (including filename) for the saved output figure.
output_figure = "/path/to/output/histogram_plot.pdf"

# Gray-value ranges for each material class, used to draw the shaded overlay
# bands. Adjust these thresholds to match the segmentation ranges of your
# dataset. Values are specific to the imaging session and scanner calibration.
root_shade_min = 5742    # Lower bound of the root gray-value range
root_shade_max = 6648    # Upper bound of the root gray-value range
sand_shade_min = 5410    # Lower bound of the sand gray-value range
sand_shade_max = 9653    # Upper bound of the sand gray-value range

# X-axis display range. Set to cover the gray-value region of interest
# while excluding uninformative tails of the distribution.
xlim_min = 2500
xlim_max = 25000

# ---------------------------------------------------------------------------
# Load histogram CSVs
# ---------------------------------------------------------------------------
# The full histogram CSV is assumed to have a header row; columns are renamed
# for consistency. Root and sand CSVs are assumed to have no header row.
hist_df = pd.read_csv(histogram_csv)
hist_df.columns = ['Intensity', 'Count']

root_df = pd.read_csv(root_csv, header=None, names=['Intensity', 'Count'])
sand_df = pd.read_csv(sand_csv, header=None, names=['Intensity', 'Count'])

# ---------------------------------------------------------------------------
# Filtering
# ---------------------------------------------------------------------------
# Remove intensity bins with very low counts (Count <= 10) to suppress noise
# and artefacts at the tails of the per-class histograms. This ensures that
# the reported min/max intensity values reflect meaningful signal rather than
# isolated outlier bins.
root_df = root_df[root_df['Count'] > 10].reset_index(drop=True)
sand_df = sand_df[sand_df['Count'] > 10].reset_index(drop=True)

# ---------------------------------------------------------------------------
# Per-class intensity range reporting
# ---------------------------------------------------------------------------
# Report the gray-value extent of each material class after noise filtering.
# These values can be used to verify or update the shading bounds above.
lowest_root  = root_df['Intensity'].min()
highest_root = root_df['Intensity'].max()
lowest_sand  = sand_df['Intensity'].min()
highest_sand = sand_df['Intensity'].max()

print(f"Root  intensity range: {lowest_root} – {highest_root}")
print(f"Sand  intensity range: {lowest_sand} – {highest_sand}")

# ---------------------------------------------------------------------------
# Cubic spline interpolation
# ---------------------------------------------------------------------------
# The raw histogram is a discrete distribution sampled at integer gray values.
# Cubic interpolation produces a smooth continuous curve for visualization,
# making peaks and valleys easier to interpret in a publication figure.
# 1000 evenly spaced points are used to render a visually smooth line.
x = hist_df['Intensity']
y = hist_df['Count']
f    = interp1d(x, y, kind='cubic')
xnew = np.linspace(x.min(), x.max(), 1000)
ynew = f(xnew)

# ---------------------------------------------------------------------------
# Plot
# ---------------------------------------------------------------------------
plt.figure(figsize=(10, 5))

# Smoothed full-volume histogram
plt.plot(xnew, ynew, color='black', label='NLM filtered histogram')

# Solid filled band marking the root gray-value range.
plt.axvspan(root_shade_min, root_shade_max, color='gold', alpha=1, label='Root')

# Hatched band marking the sand gray-value range. Using a hatch pattern
# instead of a solid fill allows the two overlapping regions to remain
# visually distinguishable. linewidth=0 suppresses the band border.
plt.rcParams['hatch.linewidth'] = 8
plt.axvspan(sand_shade_min, sand_shade_max,
            facecolor='none', edgecolor='cornflowerblue',
            hatch="/", alpha=0.7, linewidth=0, label='Sand')

# Axis labels and scale
plt.xlabel("Gray Value")
plt.ylabel("Pixel Count")

# Log scale on the y-axis compresses the large dynamic range of pixel counts,
# making both high-count peaks and low-count tails visible simultaneously.
# Comment out the next line to switch to a linear y-axis.
plt.yscale('log')
plt.ylim(10**0, 10**8)
plt.xlim(xlim_min, xlim_max)

plt.legend(frameon=False, loc='upper right')
plt.tight_layout()
plt.savefig(output_figure, dpi=300, bbox_inches="tight")
plt.show()